# Imports

In [58]:
import pandas as pd
import numpy as np
from pathlib import Path
import psycopg2
from sqlalchemy import create_engine
from dotenv import load_dotenv

import os
import sqlite3
import json

# Load Data

In [3]:
# db connection
# conn = sqlite3.connect("../data_dashboard/data/dashboard_data.sqlite")
# load_dotenv() # take environment variables from .env.
# db_user = os.getenv("POSTGRESQL_USERNAME")
# db_password = os.getenv("POSTGRESQL_PWD")
# db_host = 'localhost'
# db_port = '5432'
# db_name = 'tile_db'
# try:
#     conn = create_engine(f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')
# except:
#     print('Connection Failed')


# query = f"""
# WITH rank AS (
#     SELECT 
#         t.cluster_label,
#         tdj.date,
#         tdj.time,
#         tdj.latitude,
#         tdj.longitude,
#         w.elevation_meters_asl AS elevation,
#         w.temperature_2m AS temperature,
#         w.relative_humidity_2m AS relative_humidity,
#         w.cloud_cover,
#         w.precipitation,
#         t.tag,
#         ca.country,
#         ROW_NUMBER() OVER(PARTITION BY t.cluster_label ORDER BY tdj.date DESC, tdj.time DESC) AS rn
#     FROM tags AS t 
#     INNER JOIN tile_data_john AS tdj 
#         ON t.cluster_label = tdj.cluster_label
#     INNER JOIN weather AS w
#         ON t."index" = w."index"
#     INNER JOIN cluster_address as ca
# 		ON t.cluster_label = ca.cluster_label
#     WHERE 
#         t.tag NOT IN ('street_address','plus_code','route','premise','subpremise','establishment','point_of_interest')
# )

# SELECT
#     cluster_label,
#     date,
#     time,
#     country, 
#     latitude,
#     longitude,
#     elevation,
#     temperature,
#     relative_humidity,
#     cloud_cover,
#     precipitation,
#     tag
# FROM rank
# WHERE rn = 1;
# """

# df = pd.read_sql(query, con=conn)
# conn.dispose()
# # df = pd.read_sql_query(sql=query, con=conn)
# df

In [18]:
df = pd.read_csv('data_LLM_format.csv')
df = df.dropna(subset='area')
df

,cluster_label,date,time,area,country,latitude,longitude,elevation,temperature,relative_humidity,cloud_cover,precipitation,tag
2,1,2025-05-25,06:16:03,Auckland,New Zealand,-37.006108,174.782848,0,26.5000,78.710440,100,0.0,transit_station
3,2,2025-01-28,11:11:05,Gujarat,India,23.072692,72.617356,33,0.2730,39.008812,0,0.0,restaurant
4,6,2025-08-15,14:19:56,Islas Galapagos,Ecuador,-0.444882,-90.269456,62,32.0460,39.588930,3,0.0,airport
5,7,2025-08-03,18:01:37,Galapagos Province,Ecuador,-0.906814,-89.614383,7,31.8635,42.795937,15,0.0,airport
6,9,2025-07-24,14:22:04,Bogota,Colombia,4.698534,-74.140088,56,26.6495,81.891080,100,0.0,transit_station
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1811,3809,2025-06-01,12:13:54,Ciudad Autonoma de Buenos Aires,Argentina,-34.604218,-58.376316,31,24.4070,93.035576,90,0.0,lodging
1812,3810,2025-05-31,10:24:44,Ciudad Autonoma de Buenos Aires,Argentina,-34.604217,-58.376318,31,24.4070,81.114395,8,0.0,lodging
1813,3812,2025-06-01,01:43:24,Ciudad Autonoma de Buenos Aires,Argentina,-34.604212,-58.376299,31,28.9570,61.290936,19,0.0,lodging
1814,3815,2025-05-31,22:57:13,Ciudad Autonoma de Buenos Aires,Argentina,-34.604213,-58.376300,33,25.9860,83.311210,100,0.0,lodging


# Format Data

In [32]:
print([d for d in df['date'].sort_values().unique()])

['2024-11-11', '2024-11-13', '2024-11-14', '2024-11-15', '2024-11-16', '2024-11-17', '2024-11-18', '2024-11-19', '2024-11-20', '2024-11-21', '2024-11-22', '2024-11-23', '2024-11-24', '2024-11-25', '2024-11-26', '2024-12-27', '2024-12-28', '2024-12-29', '2024-12-30', '2024-12-31', '2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05', '2025-01-06', '2025-01-07', '2025-01-08', '2025-01-09', '2025-01-10', '2025-01-11', '2025-01-12', '2025-01-13', '2025-01-14', '2025-01-15', '2025-01-17', '2025-01-18', '2025-01-19', '2025-01-20', '2025-01-21', '2025-01-22', '2025-01-23', '2025-01-24', '2025-01-25', '2025-01-26', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31', '2025-02-01', '2025-02-02', '2025-02-03', '2025-02-05', '2025-02-06', '2025-02-07', '2025-02-08', '2025-02-09', '2025-02-10', '2025-02-11', '2025-02-12', '2025-02-13', '2025-02-14', '2025-02-15', '2025-02-16', '2025-02-17', '2025-02-18', '2025-02-19', '2025-02-20', '2025-02-21', '2025-02-23', '2025

In [15]:
df.iloc[3]

cluster_label                 2
date                 2025-01-28
time                   11:11:05
country                   India
latitude              23.072692
longitude             72.617356
elevation                  33.0
temperature               0.723
relative_humidity      36.71331
cloud_cover                 1.0
precipitation               0.0
tag                        food
Name: 3, dtype: object

In [8]:
# format by cluster
row = df.iloc[250]
text = f"""John and Maya were in {row['country']} in the {row['area']} area on {row['date']} at {row['time']} UTC.
Their location was latitude: {row['latitude']}, latitude: {row['longitude']} and has the tag: '{row['tag']}' from GoogleMaps.
"""
print(text)

John and Maya were in India in the Kerala area on 2025-01-28 at 01:49:57 UTC.
Their location was latitude: 9.462265014648438, latitude: 76.34416316245593 and has the tag: 'store' from GoogleMaps.



In [ ]:
# format by trip segment
trip_segments = {
    1: ("South Korea", ["2024-11-15", "2024-12-05"]),
    2: ("Japan", ["2024-12-05",	"2024-12-22"]),
    3: ("Thailand",	["2024-12-22", "2025-01-18"]),
    4: ("Malaysia",	["2025-01-18", "2025-01-23"]),
    5: ("India", ["2025-01-23", "2025-02-05"]),
    6: ("Vietnam", ["2025-02-05", "2025-03-11"]),
    7: ("USA", ["2025-03-11", "2025-03-18"]),
    8: ("Japan", ["2025-03-20", "2025-04-16"]),
    9: ("China", ["2025-04-16", "2025-04-25"]),
    10: ("Hong Kong", ["2025-04-25", "2025-04-29"]),
    11: ("Taiwan", ["2025-04-29", "2025-05-03"]),
    12: ("Cambodia", ["2025-05-03", "2025-05-08"]),
    13: ("Singapore", ["2025-05-08", "2025-05-11"]),
    14: ("Australia", ["2025-05-11", "2025-05-25"]),
    15: ("Argentina", ["2025-05-25", "2025-07-14"]),
    16: ("Colombia", ["2025-07-14", "2025-07-24"]),
    17: ("Ecuador", ["2025-07-24", "2025-08-15"]),
    18: ("Peru", ["2025-08-15", "2025-09-13"]),
    19: ("Brazil", ["2025-09-13", "2025-10-27"]),
    20: ("Argentina", ["2025-10-27", "2025-10-29"]),
    21: ("Chile", ["2025-10-29", "2025-11-13"]),
    22: ("Argentina", ["2025-11-13", "2025-11-25"]),
    23: ("Chile", ["2025-11-25", "2025-12-01"]),
    24: ("New Zealand", ["2025-12-01", "2025-12-22"]),
}

segment_info = {}
for segment, (country, dates) in trip_segments.items():
    entries = [f'Trip Segment {segment}: John and Maya were in {country} from {dates[0]} to {dates[1]}']
    for _, row in df.iloc[np.where((df['date'] >= dates[0]) & (df['date'] < dates[1]))].sort_values(by='date').iterrows():
        entries.append(
            f"<entry>On {row['date']} at {row['time']} UTC. They were in the {row['area']} area in {row['country']}. \
Their coordinates were ({row['latitude']}, {row['longitude']}) \
at a location with the tag: '{row['tag']}' from GoogleMaps.<end_entry>"
        )
    segment_info[segment] = "\n".join(entries)
print(segment_info[13])
    

Trip Segment 13: John and Maya were in Singapore from 2025-05-08 to 2025-05-11
<entry>On 2025-05-08 at 03:04:45 UTC. They were in the Siem Reap Province area in Cambodia. Their coordinates were (13.374685, 104.221531) at a location with the tag: 'store' from GoogleMaps.<end_entry>
<entry>On 2025-05-08 at 01:27:43 UTC. They were in the Siem Reap Province area in Cambodia. Their coordinates were (13.37674265364619, 104.22034699188164) at a location with the tag: 'airport' from GoogleMaps.<end_entry>
<entry>On 2025-05-08 at 02:18:09 UTC. They were in the Siem Reap Province area in Cambodia. Their coordinates were (13.37549078010414, 104.21990912327522) at a location with the tag: 'bar' from GoogleMaps.<end_entry>
<entry>On 2025-05-08 at 00:23:40 UTC. They were in the Siem Reap Province area in Cambodia. Their coordinates were (13.350051695044046, 103.85718624451806) at a location with the tag: 'veterinary_care' from GoogleMaps.<end_entry>
<entry>On 2025-05-08 at 00:14:38 UTC. They were in

# Load Formatted Data

In [77]:
with open("finetune_data/joya_journal.json", 'r') as f:
    journal = json.load(f)
journal[0]

{'Trip Segment': 1,
 'Date': '2024-11-15',
 'Country': ['South Korea'],
 'Entry': "Today's the day! Maya and I, as Joya, have officially started our gap year adventure. We landed at Incheon International Airport and took the train to our lodging near Bukchon Hanok Village in Jongno-gu, Seoul. I already love the mix of old and new—traditional Korean houses surrounded by the modern city. We settled into our traditional guesthouse and got a good night's rest after the long journey."}

In [87]:
meta = {}
for j in journal:
    if j['Country'][0] in meta.keys():
        meta[j['Country'][0]]['count'] += 1
        meta[j['Country'][0]]['entries'].append(j)
    else:
        meta[j['Country'][0]] = {'count':1, 'entries': [j]}
meta['Japan']

{'count': 42,
 'entries': [{'Trip Segment': 2,
   'Date': '2024-12-05',
   'Country': ['Japan'],
   'Entry': "We finally arrived in Osaka, Japan, from Busan, South Korea! We checked into our hotel and immediately set out to explore the neighborhood. We ended up in Dotonbori, which is even more vibrant and dazzling than the pictures. The energy is incredible, and we just had to try some street food. It was the perfect welcome to Japan. I have a feeling we'll be spending a lot of time here."},
  {'Trip Segment': 2,
   'Date': '2024-12-06',
   'Country': ['Japan'],
   'Entry': "Today was a deep dive into Osaka's history. We started at the majestic Osaka Castle, which is absolutely stunning. Walking through the grounds and the main tower gave us a great sense of the city's past. We continued our history lesson at the Osaka Museum of History and then had a somber but important visit to Peace Osaka, the city's museum dedicated to World War II. For dinner, we went back to Dotonbori for more d

# Generate Question/Answer Pairs

In [111]:
import google.generativeai as genai
from dotenv import load_dotenv
import os
from ast import literal_eval

In [ ]:

load_dotenv() # take environment variables from .env.
api_key = os.getenv("GOOGLE_API_KEY")
# Configure your API key
genai.configure(api_key=api_key)
model_name = 'gemini-2.0-flash'

In [98]:
journal[:2]

[{'Trip Segment': 1,
  'Date': '2024-11-15',
  'Country': ['South Korea'],
  'Entry': "Today's the day! Maya and I, as Joya, have officially started our gap year adventure. We landed at Incheon International Airport and took the train to our lodging near Bukchon Hanok Village in Jongno-gu, Seoul. I already love the mix of old and new—traditional Korean houses surrounded by the modern city. We settled into our traditional guesthouse and got a good night's rest after the long journey."},
 {'Trip Segment': 1,
  'Date': '2024-11-16',
  'Country': ['South Korea'],
  'Entry': "We spent most of the day exploring the area right around our guesthouse in Bukchon Hanok Village. The area is stunning, full of beautifully preserved hanok houses. We walked the charming, hilly streets, admiring the architecture and found a cozy spot for a coffee and some light shopping. We decided to take it easy today to get over the jet lag and are excited for tomorrow's palace visits."}]

In [126]:
pairs = []
for j in range(10, len(journal)+10, 10):
    print(j)
    if j==290:
        j = 287-1
        context = journal[280:286]
    else:
        context = journal[j-10:j]
    prompt_text = f"""
    [Prompt]
    You are an expert data generation system. Your task is to generate high-quality, contextual question-answer pairs for fine tuning a travel chatbot named Joya. Joya's knowledge base is the provided trip journal entries. Generate 20 Question-Answer pairs.
    Constraints for Generation:
    Question Format: The questions must be natural, conversational, and something a user would ask a chatbot (e.g., "Tell me about..." or "What did you do on...").
    Answer Format (Joya's Voice): The answers must be accurate, drawn only from the provided journal data, and written in the first person (from Joya's perspective, using "I," "we," "my," "our").
    Output Format: Provide the results as a clean list of Python dictionaries with the keys prompt (for the question) and completion (for Joya's answer).
    [End Prompt]

    [Context]
    {context}
    [End Context]

    """
    
    response = genai.GenerativeModel(model_name).generate_content(prompt_text)

    with open("finetune_data/gemini_responses.txt", 'a') as f:
        f.write(response.text + '\n')
    qa_pair = literal_eval(response.text.replace('```python\n','').replace('\n```',''))
    with open("finetune_data/QA_pairs.txt", 'a') as f:
        f.write(str(qa_pair) + '\n')
    pairs += qa_pair

10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290


In [127]:
len(pairs)

580